In [1]:
# Import các thư viện cần thiết cho việc crawling dữ liệu cryptocurrency
import websocket  # Thư viện để kết nối WebSocket với Bitget API
import json  # Thư viện để xử lý dữ liệu JSON từ API
import csv  # Thư viện để ghi dữ liệu vào file CSV
import datetime  # Thư viện để xử lý thời gian
import os  # Thư viện để thao tác với file và thư mục
import pandas as pd  # Thư viện để xử lý dữ liệu dạng bảng
import threading  # Thư viện để chạy WebSocket trong background thread
import time  # Thư viện để xử lý sleep và timing
from collections import deque  # Cấu trúc dữ liệu queue hiệu quả
from collections import defaultdict  # Dictionary với giá trị mặc định

In [2]:
# Định nghĩa các hằng số thời gian (tính bằng giây)
a=60  # 1 phút = 60 giây
b=a*60  # 1 giờ = 60 phút = 3600 giây
c=b*24  # 1 ngày = 24 giờ = 86400 giây
d=c*7  # 1 tuần = 7 ngày
e=c*30  # 1 tháng = 30 ngày (ước tính)

In [3]:
# Cấu hình kết nối WebSocket và các symbol cần theo dõi
WEBSOCKET_URL =  "wss://ws.bitget.com/v2/ws/public"  # URL WebSocket public của Bitget
INSTRUMENT_IDS = ["SOLUSDT", "BTCUSDT", "ETHUSDT"]  # Danh sách các cặp tiền cần crawl dữ liệu
TRADE_CSV_FILE = "trade_data.csv"  # Tên file CSV để lưu dữ liệu giao dịch

In [4]:
# Hàm tạo file CSV với cấu trúc header chuẩn
def tao_file_csv():
    # Định nghĩa các cột dữ liệu cho file CSV
    header = [
    "thoi_gian",          # Thời gian hệ thống ghi nhận dữ liệu (local time)
    "timestamp_api",      # Timestamp từ API Bitget (ts)
    "instId",             # Mã sản phẩm giao dịch (ví dụ: BTCUSDT, ETHUSDT)
    "trade_id",           # ID duy nhất của giao dịch từ exchange
     "price",              # Giá giao dịch tại thời điểm đó
    "size",               # Khối lượng/số lượng coin được giao dịch
    "side",               # Hướng giao dịch (buy=mua, sell=bán)
    "action"              # Loại dữ liệu (snapshot=dữ liệu cũ, update=dữ liệu mới)
]

    # Kiểm tra xem file CSV đã tồn tại chưa hoặc file rỗng
    if not os.path.exists(TRADE_CSV_FILE) or os.path.getsize(TRADE_CSV_FILE) == 0:
        # Tạo file mới và ghi header
        with open(TRADE_CSV_FILE, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(header)
        print(f"Đã tạo file CSV với {len(header)} cột: {TRADE_CSV_FILE}")
    else:
        print(f"File CSV đã tồn tại: {TRADE_CSV_FILE}")

# Gọi hàm để tạo file CSV ngay khi chạy
tao_file_csv()

Đã tạo file CSV với 8 cột: trade_data.csv


In [5]:
# Hàm được gọi khi WebSocket kết nối thành công
def on_open(ws):
    print(f" Đã kết nối thành công")
    # Tạo thread riêng để reset bộ đếm trades mỗi phút
    reset_thread = threading.Thread(target=reset_counters, daemon=True)
    reset_thread.start()

    # Đăng ký theo dõi channel "trade" cho từng symbol
    for inst_id in INSTRUMENT_IDS:
        subscribe_message = {
            "op": "subscribe",  # Lệnh đăng ký
            "args": [
                {
                    "instType": "SPOT",      # Loại sản phẩm: giao dịch spot
                    "channel": "trade",      # Channel theo dõi: dữ liệu giao dịch
                    "instId": inst_id        # Symbol cần theo dõi
                }
            ]
        }
        ws.send(json.dumps(subscribe_message))
        print(f"Đang theo dõi {inst_id}")
    print(f"Đang theo dõi các cặp: {', '.join(INSTRUMENT_IDS)}")

# Biến toàn cục để lưu trữ dữ liệu
all_data= []  # Lưu tất cả dữ liệu thô từ WebSocket
trade_counters = defaultdict(int)  # Đếm số trades cho mỗi symbol (tự động khởi tạo = 0)
MAX_TRADES_PER_SYMBOL = 50  # Giới hạn tối đa 50 trades mỗi symbol mỗi phút

# Hàm reset bộ đếm trades mỗi phút để tránh spam
def reset_counters():
    while True:
        time.sleep(60)  # Chờ 60 giây (1 phút)
        global trade_counters
        old_counters = dict(trade_counters)  # Lưu lại số liệu cũ để thống kê
        trade_counters.clear()  # Xóa bộ đếm, reset về 0
        print(f" Reset counters sau 1 phút. Trades đã lưu: {old_counters}")

# Hàm xử lý tin nhắn từ WebSocket (hàm quan trọng nhất)
def on_message(ws, message_str):  
    global all_data, trade_counters
    # Parse JSON từ tin nhắn nhận được
    data = json.loads(message_str)
    all_data.append(data)  # Lưu dữ liệu thô vào mảng
    
    print(f"Received: {data}")
    
    # Xử lý tin nhắn đăng ký thành công (không phải dữ liệu trade)
    if "event" in data:
        print(f"Subscription event: {data.get('event')} for {data.get('arg', {}).get('instId')}")
        return
    
    # Xử lý dữ liệu trade thực tế
    if "data" in data and data["data"]:
        instId = data.get("arg", {}).get("instId")  # Lấy symbol (VD: BTCUSDT)
        action = data.get('action', 'unknown')  # Lấy loại action (snapshot/update)
        
        # Bỏ qua snapshot data để tránh spam (có thể bỏ qua nếu không cần thiết)
        # if action == 'snapshot':
        #     print(f" Bỏ qua snapshot data cho {instId}")
        #     return
        
        # Kiểm tra giới hạn trades cho symbol này
        if trade_counters[instId] >= MAX_TRADES_PER_SYMBOL:
            print(f" {instId} đã đạt giới hạn {MAX_TRADES_PER_SYMBOL} trades trong phút này")
            return
        
        # Xử lý từng trade trong dữ liệu (có thể có nhiều trades trong 1 message)
        for trade in data["data"]:
            # Kiểm tra lại giới hạn cho từng trade
            if trade_counters[instId] >= MAX_TRADES_PER_SYMBOL:
                print(f" {instId} đã đạt giới hạn {MAX_TRADES_PER_SYMBOL} trades")
                break
                
            # Trích xuất dữ liệu từ trade
            thoi_gian = datetime.datetime.now().isoformat()  # Thời gian hiện tại của hệ thống
            timestamp_api = trade.get('ts')  # Timestamp từ API
            trade_id = trade.get('tradeId')  # ID giao dịch
            price = trade.get('price')  # Giá giao dịch
            size = trade.get('size')  # Khối lượng
            side = trade.get('side')  # Hướng giao dịch (buy/sell)
            
            # Tăng bộ đếm cho symbol này
            trade_counters[instId] += 1
            
            # In thông tin trade ra console để theo dõi
            print(f" {instId} [{trade_counters[instId]}/{MAX_TRADES_PER_SYMBOL}] | Giá: {price} | Size: {size} | Side: {side}")
            
            # Lưu dữ liệu vào file CSV
            with open(TRADE_CSV_FILE, mode='a', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                writer.writerow([thoi_gian, timestamp_api, instId, trade_id, price, size, side, action])
    else:
        pass  # Bỏ qua các message không có dữ liệu

# Hàm xử lý lỗi WebSocket
def on_error(ws, error):
    print(f" Lỗi: {error}")

# Hàm xử lý khi WebSocket đóng kết nối
def on_close(ws, close_status_code, close_msg):
    print(f" Kết nối đã đóng - Code: {close_status_code}")
    # In thống kê cuối cùng về số trades đã crawl
    print(" Thống kê cuối cùng:")
    for symbol, count in trade_counters.items():
        print(f" {symbol}: {count} trades")

In [6]:
# Hàm chạy WebSocket trong thread riêng
def run_ws():
    # Chạy WebSocket với ping mỗi 30 giây, timeout 10 giây
    ws.run_forever(ping_interval=30, ping_timeout=10)

# Tạo đối tượng WebSocket với các callback functions
ws = websocket.WebSocketApp(WEBSOCKET_URL,
                          on_open=on_open,      # Gọi khi kết nối thành công
                          on_message=on_message,  # Gọi khi nhận tin nhắn
                          on_error=on_error,    # Gọi khi có lỗi
                          on_close=on_close)    # Gọi khi đóng kết nối

# Thông báo bắt đầu crawling
print(f"Bắt đầu kết nối đến Bitget...")
print(f"Dữ liệu sẽ được lưu vào: {TRADE_CSV_FILE}")
print("Nhấn Ctrl+C để dừng")

# Tạo thread riêng để chạy WebSocket (không block main thread)
ws_thread = threading.Thread(target=run_ws)
ws_thread.daemon = True  # Thread sẽ tự động tắt khi main thread tắt
ws_thread.start()

# Thời gian chạy = a*6 = 60*6 = 360 giây = 6 phút
run_duration = a*6

# Chạy chương trình trong khoảng thời gian định trước
try:
    time.sleep(run_duration)  # Sleep trong 6 phút
except KeyboardInterrupt:
    print("\nĐã dừng bằng Ctrl+C")  # Xử lý khi user nhấn Ctrl+C

# Đóng kết nối WebSocket và thông báo
ws.close()
print(f"Đã ngắt kết nối sau {run_duration} giây.")

Bắt đầu kết nối đến Bitget...
Dữ liệu sẽ được lưu vào: trade_data.csv
Nhấn Ctrl+C để dừng
 Đã kết nối thành công
Đang theo dõi SOLUSDT
Đang theo dõi BTCUSDT
Đang theo dõi ETHUSDT
Đang theo dõi các cặp: SOLUSDT, BTCUSDT, ETHUSDT
Received: {'event': 'subscribe', 'arg': {'instType': 'SPOT', 'channel': 'trade', 'instId': 'SOLUSDT'}}
Subscription event: subscribe for SOLUSDT
Received: {'event': 'subscribe', 'arg': {'instType': 'SPOT', 'channel': 'trade', 'instId': 'BTCUSDT'}}
Subscription event: subscribe for BTCUSDT
Received: {'event': 'subscribe', 'arg': {'instType': 'SPOT', 'channel': 'trade', 'instId': 'ETHUSDT'}}
Subscription event: subscribe for ETHUSDT
Received: {'action': 'snapshot', 'arg': {'instType': 'SPOT', 'channel': 'trade', 'instId': 'SOLUSDT'}, 'data': [{'ts': '1749279174425', 'price': '152.21', 'size': '0.7656', 'side': 'buy', 'tradeId': '1315137666408718437'}, {'ts': '1749279167917', 'price': '152.21', 'size': '956.1800', 'side': 'buy', 'tradeId': '1315137639112187905'}, {

 Reset counters sau 1 phút. Trades đã lưu: {}
 Reset counters sau 1 phút. Trades đã lưu: {}
 Reset counters sau 1 phút. Trades đã lưu: {}
 Reset counters sau 1 phút. Trades đã lưu: {}
 Reset counters sau 1 phút. Trades đã lưu: {}
 Reset counters sau 1 phút. Trades đã lưu: {}
 Reset counters sau 1 phút. Trades đã lưu: {}
 Reset counters sau 1 phút. Trades đã lưu: {}
 Reset counters sau 1 phút. Trades đã lưu: {}
 Reset counters sau 1 phút. Trades đã lưu: {}
 Reset counters sau 1 phút. Trades đã lưu: {}
 Reset counters sau 1 phút. Trades đã lưu: {}
 Reset counters sau 1 phút. Trades đã lưu: {}
 Reset counters sau 1 phút. Trades đã lưu: {}
 Reset counters sau 1 phút. Trades đã lưu: {}
 Reset counters sau 1 phút. Trades đã lưu: {}
 Reset counters sau 1 phút. Trades đã lưu: {}
 Reset counters sau 1 phút. Trades đã lưu: {}
 Reset counters sau 1 phút. Trades đã lưu: {}
